<a href="https://colab.research.google.com/github/DenizhanOngun/463Project/blob/main/463_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
IMDB Sentiment Classification – Data Loading & Tokenization Pipeline
=====================================================================
Supports RoBERTa and DeBERTa as backbone encoders for an ensemble network.

Backbone options
----------------
  roberta-base          (125M params)
  roberta-large         (355M params)
  microsoft/deberta-v3-base   (86M params)
  microsoft/deberta-v3-large  (304M params)

Usage
-----
  python imdb_data_pipeline.py --backbone roberta-base --max_length 256 --batch_size 32
  python imdb_data_pipeline.py --backbone microsoft/deberta-v3-base
"""

import argparse
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from datasets import load_dataset, DatasetDict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, PreTrainedTokenizerBase

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

SUPPORTED_BACKBONES: Dict[str, str] = {
    "roberta-base": "roberta-base",
    "roberta-large": "roberta-large",
    "deberta-v3-base": "microsoft/deberta-v3-base",
    "deberta-v3-large": "microsoft/deberta-v3-large",
}

LABEL_MAP: Dict[int, str] = {0: "negative", 1: "positive"}


@dataclass
class PipelineConfig:
    """All hyper-parameters for data loading and tokenization."""

    backbone: str = "roberta-base"
    max_length: int = 256
    batch_size: int = 32
    num_workers: int = 4
    val_split: float = 0.1          # fraction of train set used for validation
    seed: int = 42
    cache_dir: Optional[str] = None # set to a local path to cache HuggingFace data
    truncation_strategy: str = "longest_first"
    padding_strategy: str = "max_length"

    # Fields filled automatically – do not set manually
    tokenizer_name: str = field(init=False)

    def __post_init__(self) -> None:
        # Resolve short alias → full HuggingFace model id
        self.tokenizer_name = SUPPORTED_BACKBONES.get(self.backbone, self.backbone)
        logger.info("Pipeline config: backbone=%s | max_length=%d | batch_size=%d",
                    self.tokenizer_name, self.max_length, self.batch_size)


# ---------------------------------------------------------------------------
# Dataset wrapper
# ---------------------------------------------------------------------------

class IMDBTokenizedDataset(Dataset):
    """
    Wraps a HuggingFace IMDB split and returns tokenizer outputs
    ready to feed into a transformer encoder.

    Each item is a dict with keys
      input_ids       – (max_length,)  LongTensor
      attention_mask  – (max_length,)  LongTensor
      token_type_ids  – (max_length,)  LongTensor  [only for DeBERTa]
      label           – scalar          LongTensor  {0: neg, 1: pos}
    """

    def __init__(
        self,
        texts: List[str],
        labels: List[int],
        tokenizer: PreTrainedTokenizerBase,
        config: PipelineConfig,
    ) -> None:
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.config = config

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.config.max_length,
            padding=self.config.padding_strategy,
            truncation=self.config.truncation_strategy,
            return_tensors="pt",
            return_token_type_ids=True,   # harmless for RoBERTa (all zeros)
            return_attention_mask=True,
        )

        item = {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

        # token_type_ids are meaningful for DeBERTa; include when present
        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        return item


# ---------------------------------------------------------------------------
# Core pipeline
# ---------------------------------------------------------------------------

class IMDBDataPipeline:
    """
    End-to-end data pipeline:
      1. Downloads / caches the IMDB dataset via HuggingFace `datasets`.
      2. Splits the official train set into train / validation subsets.
      3. Loads the correct tokenizer for the chosen backbone.
      4. Returns typed DataLoaders ready for model training.

    Example
    -------
    >>> config = PipelineConfig(backbone="roberta-base", max_length=256)
    >>> pipeline = IMDBDataPipeline(config)
    >>> loaders = pipeline.get_dataloaders()
    >>> train_loader, val_loader, test_loader = loaders
    """

    def __init__(self, config: PipelineConfig) -> None:
        self.config = config
        self.tokenizer: Optional[PreTrainedTokenizerBase] = None
        self._raw: Optional[DatasetDict] = None

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def load_raw_data(self) -> DatasetDict:
        """Download (or load from cache) the IMDB dataset."""
        logger.info("Loading IMDB dataset …")
        self._raw = load_dataset(
            "imdb",
            cache_dir=self.config.cache_dir,
        )
        logger.info(
            "IMDB loaded – train: %d | test: %d",
            len(self._raw["train"]),
            len(self._raw["test"]),
        )
        return self._raw

    def load_tokenizer(self) -> PreTrainedTokenizerBase:
        """Instantiate the tokenizer that matches the chosen backbone."""
        logger.info("Loading tokenizer: %s", self.config.tokenizer_name)
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_name,
            use_fast=True,
        )
        logger.info(
            "Tokenizer ready – vocab size: %d | special tokens: %s",
            self.tokenizer.vocab_size,
            self.tokenizer.all_special_tokens,
        )
        return self.tokenizer

    def build_datasets(
        self,
    ) -> Tuple[IMDBTokenizedDataset, IMDBTokenizedDataset, IMDBTokenizedDataset]:
        """
        Build train / validation / test IMDBTokenizedDataset instances.

        The official 25 000-sample train split is subdivided into
        (1 - val_split) × 25 000 training examples and
        val_split × 25 000 validation examples using a stratified seed split.
        """
        if self._raw is None:
            self.load_raw_data()
        if self.tokenizer is None:
            self.load_tokenizer()

        # Stratified train / val split
        train_val = self._raw["train"].train_test_split(
            test_size=self.config.val_split,
            seed=self.config.seed,
            stratify_by_column="label",
        )

        def _make(split_name: str) -> IMDBTokenizedDataset:
            if split_name == "test":
                hf_split = self._raw["test"]
            elif split_name == "train":
                hf_split = train_val["train"]
            else:                           # val
                hf_split = train_val["test"]

            return IMDBTokenizedDataset(
                texts=hf_split["text"],
                labels=hf_split["label"],
                tokenizer=self.tokenizer,
                config=self.config,
            )

        train_ds = _make("train")
        val_ds   = _make("val")
        test_ds  = _make("test")

        logger.info(
            "Dataset sizes – train: %d | val: %d | test: %d",
            len(train_ds), len(val_ds), len(test_ds),
        )
        return train_ds, val_ds, test_ds

    def get_dataloaders(
        self,
    ) -> Tuple[DataLoader, DataLoader, DataLoader]:
        """
        Build and return (train_loader, val_loader, test_loader).

        The train loader shuffles; val and test loaders do not.
        """
        train_ds, val_ds, test_ds = self.build_datasets()

        def _loader(ds: IMDBTokenizedDataset, shuffle: bool) -> DataLoader:
            return DataLoader(
                ds,
                batch_size=self.config.batch_size,
                shuffle=shuffle,
                num_workers=self.config.num_workers,
                pin_memory=torch.cuda.is_available(),
            )

        train_loader = _loader(train_ds, shuffle=True)
        val_loader   = _loader(val_ds,   shuffle=False)
        test_loader  = _loader(test_ds,  shuffle=False)

        logger.info(
            "DataLoaders ready – train batches: %d | val batches: %d | test batches: %d",
            len(train_loader), len(val_loader), len(test_loader),
        )
        return train_loader, val_loader, test_loader

    # ------------------------------------------------------------------
    # Inspection helpers
    # ------------------------------------------------------------------

    def inspect_batch(self, loader: DataLoader, n: int = 2) -> None:
        """Print the shapes and decoded text for the first *n* examples."""
        batch = next(iter(loader))
        logger.info("--- Batch inspection (first %d examples) ---", n)
        for i in range(min(n, batch["input_ids"].shape[0])):
            ids    = batch["input_ids"][i]
            label  = LABEL_MAP[batch["label"][i].item()]
            tokens = self.tokenizer.convert_ids_to_tokens(ids)
            # Strip padding for display
            non_pad = (ids != self.tokenizer.pad_token_id).sum().item()
            logger.info(
                "  [%d] label=%s | seq_len=%d | tokens[:8]=%s",
                i, label, non_pad, tokens[:8],
            )

        logger.info("  input_ids shape    : %s", tuple(batch["input_ids"].shape))
        logger.info("  attention_mask shape: %s", tuple(batch["attention_mask"].shape))
        if "token_type_ids" in batch:
            logger.info("  token_type_ids shape: %s", tuple(batch["token_type_ids"].shape))


# ---------------------------------------------------------------------------
# Ensemble hook (stub – wired up in Part 2)
# ---------------------------------------------------------------------------

class EnsembleInputAdapter:
    """
    Adapter that formats a tokenized batch for two backbone encoders.

    In the full ensemble each encoder receives the same tokenized input;
    their [CLS] representations are later concatenated or mixed before
    the classification head.

    Usage (Part 2):
        adapter  = EnsembleInputAdapter(backbone_a="roberta-base",
                                        backbone_b="microsoft/deberta-v3-base")
        tok_a, tok_b = adapter.prepare(batch)
        out_a = encoder_a(**tok_a)
        out_b = encoder_b(**tok_b)
        cls_a = out_a.last_hidden_state[:, 0, :]   # (B, H_a)
        cls_b = out_b.last_hidden_state[:, 0, :]   # (B, H_b)
        joint = torch.cat([cls_a, cls_b], dim=-1)  # feed to classifier head
    """

    _ENCODER_KEYS = {"input_ids", "attention_mask", "token_type_ids"}

    def __init__(self, backbone_a: str, backbone_b: str) -> None:
        self.backbone_a = backbone_a
        self.backbone_b = backbone_b
        logger.info(
            "EnsembleInputAdapter – backbone_a: %s | backbone_b: %s",
            backbone_a, backbone_b,
        )

    def prepare(
        self,
        batch: Dict[str, torch.Tensor],
    ) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
        """
        Returns two encoder-ready dicts from a single tokenized batch.
        RoBERTa silently ignores token_type_ids; DeBERTa uses them.
        """
        keys = {k: v for k, v in batch.items() if k in self._ENCODER_KEYS}
        return keys, keys          # same tokens feed both encoders


# ---------------------------------------------------------------------------
# CLI entry-point
# ---------------------------------------------------------------------------

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="IMDB data pipeline – Part 1: loading & tokenization"
    )
    parser.add_argument(
        "--backbone",
        default="roberta-base",
        choices=list(SUPPORTED_BACKBONES.keys()) + list(SUPPORTED_BACKBONES.values()),
        help="Transformer backbone for the ensemble (default: roberta-base)",
    )
    parser.add_argument("--max_length",  type=int,   default=256)
    parser.add_argument("--batch_size",  type=int,   default=32)
    parser.add_argument("--num_workers", type=int,   default=4)
    parser.add_argument("--val_split",   type=float, default=0.1)
    parser.add_argument("--seed",        type=int,   default=42)
    parser.add_argument("--cache_dir",   type=str,   default=None)
    return parser.parse_args()


def main() -> None:
    args = parse_args()

    config = PipelineConfig(
        backbone=args.backbone,
        max_length=args.max_length,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        val_split=args.val_split,
        seed=args.seed,
        cache_dir=args.cache_dir,
    )

    pipeline = IMDBDataPipeline(config)
    train_loader, val_loader, test_loader = pipeline.get_dataloaders()

    # Quick sanity-check
    pipeline.inspect_batch(train_loader, n=2)

    # Show the ensemble adapter stub
    adapter = EnsembleInputAdapter(
        backbone_a=config.tokenizer_name,
        backbone_b="microsoft/deberta-v3-base",
    )
    batch = next(iter(train_loader))
    tok_a, tok_b = adapter.prepare(batch)
    logger.info("Ensemble adapter – tok_a keys: %s | tok_b keys: %s",
                list(tok_a.keys()), list(tok_b.keys()))


if __name__ == "__main__":
    main()